<a href="https://colab.research.google.com/github/maulikcmr05/NLP/blob/main/1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk import pos_tag
from nltk.corpus import wordnet
from transformers import pipeline

In [14]:
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("averaged_perceptron_tagger")
nltk.download("averaged_perceptron_tagger_eng")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [15]:
def get_wordnet_pos(tag):
  if tag.startswith("J"):
    return wordnet.ADJ
  elif tag.startswith("V"):
    return wordnet.VERB
  elif tag.startswith("N"):
    return wordnet.NOUN
  elif tag.startswith("R"):
    return wordnet.ADV
  else:
    return wordnet.NOUN

In [16]:
text = input("Enter a sentence: ")

print("\n========== TOKENIZATION ==========")
tokens = word_tokenize(text)
print(tokens)

Enter a sentence: The quick brown fox jumps over the lazy dog.

========== TOKENIZATION ==========
['The', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog', '.']


In [17]:
print("\n========== MORPHOLOGICAL ANALYSIS ==========")
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()
pos_tags = pos_tag(tokens)
print("{:<15}{:<15}{:<15}".format("Word", "Stem", "Lemma"))

print("-" * 45)
for word, tag in pos_tags:
  stem = stemmer.stem(word)
  lemma = lemmatizer.lemmatize(word, get_wordnet_pos(tag))
  print("{:<15}{:<15}{:<15}".format(word, stem, lemma))


========== MORPHOLOGICAL ANALYSIS ==========
Word           Stem           Lemma          
---------------------------------------------
The            the            The            
quick          quick          quick          
brown          brown          brown          
fox            fox            fox            
jumps          jump           jump           
over           over           over           
the            the            the            
lazy           lazi           lazy           
dog            dog            dog            
.              .              .              


In [18]:
print("\n========== SYNTACTIC ANALYSIS ==========")
print("{:<15}{:<10}".format("Word", "POS Tag"))
print("-" * 30)
for word, tag in pos_tags:
  print("{:<15}{:<10}".format(word, tag))


========== SYNTACTIC ANALYSIS ==========
Word           POS Tag   
------------------------------
The            DT        
quick          JJ        
brown          NN        
fox            NN        
jumps          VBZ       
over           IN        
the            DT        
lazy           JJ        
dog            NN        
.              .         


In [19]:
print("\n========== SEMANTIC ANALYSIS ==========")
for word in tokens:
  synsets = wordnet.synsets(word)
  if synsets:
      print(f"{word:<15} : {synsets[0].definition()}")
  else:
    print(f"{word:<15} : No meaning found")


========== SEMANTIC ANALYSIS ==========
The             : No meaning found
quick           : any area of the body that is highly sensitive to pain (as the flesh underneath the skin or a fingernail or toenail)
brown           : an orange of low brightness and saturation
fox             : alert carnivorous mammal with pointed muzzle and ears and a bushy tail; most are predators that do not hunt in packs
jumps           : a sudden and decisive increase
over            : (cricket) the division of play during which six balls are bowled at the batsman by one player from the other team from the same end of the pitch
the             : No meaning found
lazy            : moving slowly and gently
dog             : a member of the genus Canis (probably descended from the common wolf) that has been domesticated by man since prehistoric times; occurs in many breeds
.               : No meaning found


In [20]:
print("\n========== PRAGMATIC ANALYSIS ==========")
# Load model only once
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)
sentence = text.strip().lower()
# Keywords
request_words = [
    "please", "could you", "can you",
    "would you", "will you",
    "kindly", "may i", "would you mind"
]

command_words = [
    "open", "close", "run", "execute",
    "print", "show", "display",
    "write", "train", "stop",
    "start", "read", "save",
    "delete", "install",
    "create", "calculate",
    "find", "search"
]


========== PRAGMATIC ANALYSIS ==========


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

In [21]:
# Rule-based classification
if sentence.endswith("?"):
  if any(word in sentence for word in request_words):
    intent = "Request"
    confidence = 1.0000
  else:
      intent = "Question"
      confidence = 1.0000
elif any(sentence.startswith(word) for word in command_words):
  intent = "Command"
  confidence = 1.0000
elif any(word in sentence for word in request_words):
  intent = "Request"
  confidence = 1.0000
else:
  # Zero-shot fallback
  labels = [
      "This sentence is a statement.",
      "This sentence is a question.",
      "This sentence is a request.",
      "This sentence is a command."
      ]
  result = classifier(
      text,
      candidate_labels=labels,
      hypothesis_template="{}"
      )
  intent = result["labels"][0].replace("This sentence is a ","").replace(".", "").title()
  confidence = result["scores"][0]

print("Predicted Intent :", intent)
print("Confidence Score :", round(confidence, 4))

Predicted Intent : Statement
Confidence Score : 0.5326
